# EGU26 Short Course: Machine-Learning Downscaling with `idownscale`

This notebook is the practical companion to the EGU26 short course.
Release compatibility: this notebook is maintained against `idownscale` `v1.4.0`. If you use another release, re-check workflow runner names, paths, and expected outputs.

Recommended checkout for the maintained course material:

```bash
git clone https://github.com/cerfacs-globc/idownscale.git
cd idownscale
git checkout v1.4.0
```

This notebook covers the course workflow subset, not every capability available in the wider release.


It is designed to tell a complete, reproducible story:

1. fetch the upstream climate data from official repositories
2. place the files into the local `idownscale` repo layout
3. prepare the data needed for the demonstration workflow
4. run pretrained inference on the temperature case study
5. inspect validation outputs
6. optionally rebuild the training data and retrain the model

The workflow is intentionally shown phase by phase rather than as one opaque run.
After each phase, attendees should inspect one or more diagnostic outputs before moving on.


## 1. Scope and assumptions

This notebook focuses on the temperature case study used in the short course.

The intended workflow is:

- ERA5 as the coarse predictor laboratory
- E-OBS as the high-resolution observational target over France
- CNRM-CM6-1 as the example CMIP6 global climate model
- a pretrained U-Net checkpoint for the main inference demonstration
- retraining as an optional advanced extension

For consistency with the validated workflow, the intended default scope is the same as `exp5`:

- same France domain
- same long reference / training period logic
- same climate-change framing with long historical and future windows

This means some steps are inherently slow. That is not an avoidable inconvenience; it is part of using a climate workflow honestly.

This notebook is a guided example rather than a full production recipe.


## 2. Data sources

The large upstream climate files should be fetched from their official repositories.

See:

- `HOW_TO_FETCH_UPSTREAM_DATA.md`
- `DATASETS_TO_PROVIDE.md`

Recommended sources:

- ERA5 from Copernicus / CDS
- CMIP6 from Copernicus first, ESGF second
- E-OBS from the official Copernicus / KNMI distribution


In [ ]:
# Fill these paths for your environment.
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
RAWDATA = REPO_ROOT / "rawdata"
ERA5_DIR = RAWDATA / "era5"
EOBS_DIR = RAWDATA / "eobs"
GCM_DIR = RAWDATA / "gcm"

REPO_ROOT, RAWDATA

## 3. Place the fetched files into the repo layout

The short-course workflow assumes the usual `idownscale` directory layout.

Examples:

- `rawdata/era5/tas_1d/tas_1d_<YEAR>_ERA5.nc`
- `rawdata/era5/orography_ERA5.nc`
- `rawdata/eobs/tas_ens_mean_1d_025deg_reg_v29_0e_19500101-20231231_france.nc`
- `rawdata/gcm/CNRM-CM6-1/...`

Native ERA5 files from C3S/CDS can be kept as the source files in this layout.
They are then standardized and prepared by the repo-side workflow.


## 4. Prepare the France-focused data

Two levels of preparation are relevant here:

- a notebook-grade France crop / reformatting step
- the full training-grade reconstruction path used by `build_dataset.py`

The exact final pedagogical choice for the cropping recipe can still be refined,
but the repo already provides reusable tooling such as:

- `bin/preprocessing/crop_domain.py`
- `bin/preprocessing/build_dataset.py`

Current `exp5` France domain:

- `[-6.0, 10.0, 38.0, 54.0]`

For the short course, the preparation story should be explicit:

1. fetch native upstream files
2. place them in the repo layout
3. run the preparation step with repo tooling
4. inspect a small diagnostic or metadata check before continuing

The workflow now supports this as an optional pre-Phase-1 step when the France target files are not already available locally.


In [ ]:
# Optional pre-Phase-1 target preparation through the workflow runner.

prep_phase1_example = r'''
python bin/production/run_obs_workflow.py \
  --exp exp5 \
  --steps prep_phase1
'''

# Lower-level helper used underneath by the optional workflow step.
prep_helper_example = r'''
python bin/preprocessing/prepare_exp5_france_targets.py \
  --exp exp5 \
'''

print(prep_phase1_example)
print(prep_helper_example)

## 5. Run the workflow phase by phase with the master script

The notebook should drive the workflow through `run_obs_workflow.py`, but each phase should be run separately.

That gives attendees a complete story and lets them inspect intermediate outputs instead of waiting for one large black-box run.

Recommended teaching rhythm:

1. Optional pre-Phase-1: prepare France target files
2. Phase 1: build training samples
3. Stats: compute normalization statistics
4. Train: optional, advanced extension
5. Predict: run inference
6. Metrics: daily, monthly, and VALUE-style summaries
7. Plots: inspect figures after each relevant phase

For the main classroom demonstration, pretrained inference is still the simplest first path, but it should be shown as one phase inside the larger method.


In [ ]:
# Example phase-by-phase workflow commands to adapt for the course environment.

prep_phase1_example = r'''
python bin/production/run_obs_workflow.py \
  --exp exp5 \
  --steps prep_phase1
'''

phase1_example = r'''
python bin/production/run_obs_workflow.py \
  --exp exp5 \
  --steps phase1 \
  --test-name unet_course_demo
'''

stats_example = r'''
python bin/production/run_obs_workflow.py \
  --exp exp5 \
  --steps stats \
  --test-name unet_course_demo
'''

inference_example = r'''
python bin/production/run_obs_workflow.py \
  --exp exp5 \
  --steps predict_loop \
  --test-name unet_all \
  --simu-test gcm_bc
'''

metrics_example = r'''
python bin/production/run_obs_workflow.py \
  --exp exp5 \
  --steps metrics_day,metrics_month,value_metrics,plot_metrics_day,plot_metrics_month \
  --test-name unet_all \
  --simu-test gcm_bc
'''

print(prep_phase1_example)
print(phase1_example)
print(stats_example)
print(inference_example)
print(metrics_example)

## 6. Diagnostic checkpoints after each phase

A key teaching principle for this notebook is that each phase should have at least one visible check.

Suggested checks:

- after pre-Phase-1:
  - inspect the France target files
  - verify the target grid is `64x64`
  - verify the longitude and latitude ranges are correct
- after Phase 1:
  - inspect one or two `sample_YYYYMMDD.npz` files
  - check predictor and target array shapes
  - check that masks and NaNs look sensible
- after Stats:
  - inspect `statistics.json`
  - verify variables/channels match expectations
- after Predict:
  - inspect one prediction NetCDF
  - check time coverage and target grid consistency
- after Metrics:
  - inspect daily and monthly CSV summaries
  - inspect the VALUE table
- after Plots:
  - inspect the main diagnostic figures shown in the course

Useful outputs for the course include:

- daily metrics
- monthly metrics
- VALUE-style summary
- distribution comparison figures
- monthly spatial bias / RMSE figures

These can either be generated live or shipped as precomputed examples.


## 7. Optional extension: rebuild the training data and retrain

This section is optional for the short course but important for the full story.

Attendees who want to go beyond checkpoint reuse should see that the workflow can also be used to:

1. rebuild the Phase 1 training samples
2. compute normalization statistics
3. train a new model
4. evaluate the retrained checkpoint

This becomes essential when the reference setup changes, for example with a different reanalysis.

For the course, this training block should be presented as optional but fully real: it uses the same exp5 domain and the same long-period logic as the validated workflow.


In [ ]:
# Example training-oriented workflow command.

training_example = r'''
python bin/production/run_obs_workflow.py \
  --exp exp5 \
  --steps phase1,stats,train \
  --test-name unet_course_demo
'''

print(training_example)

## 8. Next refinements before publication

Before the final public version of the short-course package, we should still confirm:

- whether we also want to expose the France-mask derivation explicitly in the notebook
- how much of the original manual preparation history we want to keep visible versus wrapping it in the new helper step
- the exact subset of years and outputs to include for the public demo, even if the default narrative remains aligned with exp5

But the overall story is already in place: fetch, prepare, optionally run the pre-Phase-1 target step, run each phase separately, inspect diagnostics after each phase, and optionally retrain.
